In [ ]:
%load_ext watermark
%watermark -a "Paul A. Wambo" -v -p Bio,matplotlib,numpy,pandas,pyMIBiG,seaborn,sklearn,transformers,torch

In [ ]:
import os
import json

import pandas as pd
import torch

from Bio import SeqIO
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
basedir = "C:/Users/Yvael/Desktop/polo/data/input/mibig4gbk"
datafiles = [os.path.join(basedir, file) for file in os.listdir(basedir) if file.endswith(".gbk")]
print(f"Found {len(datafiles)} data files.")

In [ ]:
df = pd.DataFrame(columns=[
    "bgc_id",
    "protein_id",
    "protein_name",
    "dna_sequence",
    "protein_sequence",
])

In [ ]:
def process_file(filepath):
    rows = []
    for record in SeqIO.parse(filepath, "genbank"):
        for feature in record.features:
            if feature.type == "CDS":
                qualifiers = feature.qualifiers

                protein_id = qualifiers.get("protein_id", [pd.NA])[0]
                locus_tag = qualifiers.get("locus_tag", [pd.NA])[0]

                row = {
                    "bgc_id": record.id,
                    "protein_name": qualifiers.get("gene", [pd.NA])[0],
                    "protein_id": locus_tag if pd.isna(protein_id) else protein_id,
                    "protein_sequence": qualifiers.get("translation", [pd.NA])[0],
                    "dna_sequence": str(feature.extract(record.seq)),
                }
                rows.append(row)
    return rows

In [ ]:
from tqdm import tqdm

rows = []
for datafile in tqdm(datafiles, desc="Processing files"):
    _rows = process_file(datafile)
    rows.extend(_rows)

df = pd.DataFrame(rows)
df.head()

In [ ]:
outpath = "C:/Users/Yvael/Desktop/polo/data/output"
df.to_parquet(os.path.join(outpath, "mibig4_sequence_data.parquet"), engine="pyarrow", index=False)